<a href="https://colab.research.google.com/github/akshatsharma-x/ML/blob/main/ML_LAB_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
ML LAB-07: ENSEMBLE LEARNING
Bagging, Boosting, and Stacking Classifiers

This script implements and compares various ensemble learning techniques:
- Bagging (Bootstrap Aggregating)
- AdaBoost (Adaptive Boosting)
- Gradient Boosting
- Stacking (Stacked Generalization)
- Random Forest (Bonus)

Author: ML Lab
Date: 2026
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data and Preprocessing
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Base Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Ensemble Models
from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    RandomForestClassifier
)

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 80)
print("ML LAB-07: ENSEMBLE LEARNING")
print("Bagging, Boosting, and Stacking Classifiers")
print("=" * 80)
print()

# ============================================================================
# LOAD AND PREPARE DATA
# ============================================================================
print("LOADING DATASET")
print("-" * 80)

data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset: Breast Cancer Wisconsin (Diagnostic)")
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"Class distribution:")
print(f"  - Malignant (0): {sum(y==0)}")
print(f"  - Benign (1): {sum(y==1)}")

# Train-test split (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Feature Scaling (important for KNN in stacking)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print("✓ Features scaled using StandardScaler")

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 1: BAGGING CLASSIFIER
# ============================================================================
print("\nEXPERIMENT 1: BAGGING CLASSIFIER")
print("-" * 80)

print("\nBagging (Bootstrap Aggregating):")
print("  Concept: Reduce variance by training on different data subsets")
print("  Method:")
print("    1. Create multiple bootstrap samples (random sampling with replacement)")
print("    2. Train a base classifier on each sample")
print("    3. Combine predictions by majority voting")
print("  Benefits:")
print("    - Reduces overfitting")
print("    - Works well with unstable learners (e.g., Decision Trees)")
print("    - Can be parallelized (trains models independently)")

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Base Estimator: Decision Tree (unpruned)")
print(f"  Number of Estimators: 100")
print(f"  Bootstrap: True (sampling with replacement)")
print(f"  Max Samples: 100% (use all training data for each bag)")

print("\nTraining Bagging Classifier...")
bagging_model.fit(X_train_scaled, y_train)
y_pred_bagging = bagging_model.predict(X_test_scaled)

bagging_train_acc = accuracy_score(y_train, bagging_model.predict(X_train_scaled))
bagging_test_acc = accuracy_score(y_test, y_pred_bagging)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {bagging_train_acc:.4f} ({bagging_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {bagging_test_acc:.4f} ({bagging_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {bagging_train_acc - bagging_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_bagging = confusion_matrix(y_test, y_pred_bagging)
print(cm_bagging)
print(f"  TN: {cm_bagging[0,0]}, FP: {cm_bagging[0,1]}")
print(f"  FN: {cm_bagging[1,0]}, TP: {cm_bagging[1,1]}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bagging, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 2: BOOSTING CLASSIFIERS
# ============================================================================
print("\nEXPERIMENT 2: BOOSTING CLASSIFIERS")
print("-" * 80)

print("\nBoosting Concept:")
print("  - Trains models sequentially (cannot be parallelized)")
print("  - Each model focuses on correcting previous errors")
print("  - Combines weak learners to create a strong learner")
print("  - Reduces both bias and variance")

# ──────────────────────────────────────────────────────────────────────────
# 2A. AdaBoost (Adaptive Boosting)
# ──────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("2A. AdaBoost (Adaptive Boosting)")
print("─" * 80)

print("\nHow AdaBoost Works:")
print("  1. Initialize: Equal weights for all training samples")
print("  2. Train weak learner on weighted data")
print("  3. Calculate weighted error")
print("  4. Update sample weights:")
print("     - Increase weights of misclassified samples")
print("     - Decrease weights of correctly classified samples")
print("  5. Repeat steps 2-4 for n_estimators")
print("  6. Final prediction: Weighted majority vote")

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision stump
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Base Estimator: Decision Stump (max_depth=1)")
print(f"  Number of Estimators: 100")
print(f"  Learning Rate: 0.5")
print(f"  Algorithm: SAMME.R (real-valued predictions)")

print("\nTraining AdaBoost...")
adaboost_model.fit(X_train_scaled, y_train)
y_pred_adaboost = adaboost_model.predict(X_test_scaled)

adaboost_train_acc = accuracy_score(y_train, adaboost_model.predict(X_train_scaled))
adaboost_test_acc = accuracy_score(y_test, y_pred_adaboost)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {adaboost_train_acc:.4f} ({adaboost_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {adaboost_test_acc:.4f} ({adaboost_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {adaboost_train_acc - adaboost_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_adaboost = confusion_matrix(y_test, y_pred_adaboost)
print(cm_adaboost)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_adaboost, target_names=data.target_names))

# ──────────────────────────────────────────────────────────────────────────
# 2B. Gradient Boosting
# ──────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("2B. Gradient Boosting")
print("─" * 80)

print("\nHow Gradient Boosting Works:")
print("  1. Initialize with a simple model (predicts mean/mode)")
print("  2. Calculate residuals (actual - predicted)")
print("  3. Fit new model to predict residuals")
print("  4. Update ensemble: F(x) = F(x) + lr × new_model(x)")
print("  5. Repeat steps 2-4 for n_estimators")
print("  6. Final prediction: Sum of all models with learning rate")

gboost_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Number of Estimators: 100")
print(f"  Learning Rate: 0.1")
print(f"  Max Depth: 3")
print(f"  Subsample: 1.0 (use all samples)")

print("\nTraining Gradient Boosting...")
gboost_model.fit(X_train_scaled, y_train)
y_pred_gboost = gboost_model.predict(X_test_scaled)

gboost_train_acc = accuracy_score(y_train, gboost_model.predict(X_train_scaled))
gboost_test_acc = accuracy_score(y_test, y_pred_gboost)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {gboost_train_acc:.4f} ({gboost_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {gboost_test_acc:.4f} ({gboost_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {gboost_train_acc - gboost_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_gboost = confusion_matrix(y_test, y_pred_gboost)
print(cm_gboost)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gboost, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 3: STACKING CLASSIFIER
# ============================================================================
print("\nEXPERIMENT 3: STACKING CLASSIFIER")
print("-" * 80)

print("\nStacking (Stacked Generalization):")
print("  Concept: Learn to optimally combine diverse models")
print("  Architecture:")
print("    Level 0 (Base Models):")
print("      - Multiple diverse classifiers")
print("      - Each captures different patterns")
print("    Level 1 (Meta-Model):")
print("      - Learns from base model predictions")
print("      - Combines predictions optimally")

print("\nHow Stacking Works:")
print("  1. Split training data for cross-validation")
print("  2. Train base models on CV folds")
print("  3. Generate meta-features (base model predictions)")
print("  4. Train meta-model on meta-features")
print("  5. Final prediction: Meta-model output")

print("\nOur Stacking Architecture:")
print("  Base Models (Level 0):")
print("    1. Decision Tree - Captures non-linear patterns")
print("    2. K-Nearest Neighbors - Instance-based learning")
print("    3. Logistic Regression - Linear decision boundaries")
print("  Meta-Model (Level 1):")
print("    → Logistic Regression - Combines base predictions")

base_learners = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5  # 5-fold cross-validation for generating meta-features
)

print(f"\nConfiguration:")
print(f"  Base Learners: 3 (Decision Tree, KNN, Logistic Regression)")
print(f"  Final Estimator: Logistic Regression")
print(f"  Cross-Validation: 5-fold")
print(f"  Passthrough: False (only use base predictions)")

print("\nTraining Stacking Classifier...")
print("  [Step 1/3] Training base models with 5-fold CV...")
print("  [Step 2/3] Generating meta-features...")
print("  [Step 3/3] Training meta-model...")

stacking_model.fit(X_train_scaled, y_train)
y_pred_stacking = stacking_model.predict(X_test_scaled)

stacking_train_acc = accuracy_score(y_train, stacking_model.predict(X_train_scaled))
stacking_test_acc = accuracy_score(y_test, y_pred_stacking)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {stacking_train_acc:.4f} ({stacking_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {stacking_test_acc:.4f} ({stacking_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {stacking_train_acc - stacking_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_stacking = confusion_matrix(y_test, y_pred_stacking)
print(cm_stacking)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_stacking, target_names=data.target_names))

# Show individual base model performance
print("\n" + "─" * 80)
print("Individual Base Model Performance vs Stacking:")
print("─" * 80)

for name, model in base_learners:
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    model_name = {'dt': 'Decision Tree', 'knn': 'KNN', 'lr': 'Logistic Regression'}[name]
    print(f"  {model_name:20s}: {acc:.4f} ({acc*100:.2f}%)")

print(f"  {'Stacking (Combined)':20s}: {stacking_test_acc:.4f} ({stacking_test_acc*100:.2f}%)")
print("─" * 80)

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 4: RANDOM FOREST (BONUS)
# ============================================================================
print("\nBONUS: RANDOM FOREST CLASSIFIER")
print("-" * 80)

print("\nRandom Forest:")
print("  - Special case of Bagging with Decision Trees")
print("  - Additional randomness: Random feature subset at each split")
print("  - Each tree sees different data AND different features")
print("  - Reduces correlation between trees → better ensemble")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Number of Estimators: 100")
print(f"  Max Features: sqrt(n_features) ≈ {int(np.sqrt(X.shape[1]))}")
print(f"  Bootstrap: True")

print("\nTraining Random Forest...")
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_test, y_pred_rf)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {rf_train_acc:.4f} ({rf_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {rf_test_acc:.4f} ({rf_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {rf_train_acc - rf_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# COMPREHENSIVE COMPARISON
# ============================================================================
print("\nCOMPREHENSIVE MODEL COMPARISON")
print("=" * 80)

models = {
    'Bagging': bagging_model,
    'AdaBoost': adaboost_model,
    'GradientBoost': gboost_model,
    'Stacking': stacking_model,
    'RandomForest': rf_model
}

results = []
for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    y_pred = model.predict(X_test_scaled)

    precision = precision_score(y_test, y_pred, pos_label=0)
    recall = recall_score(y_test, y_pred, pos_label=0)
    f1 = f1_score(y_test, y_pred, pos_label=0)

    results.append({
        'Model': name,
        'Train_Acc': train_acc,
        'Test_Acc': test_acc,
        'Gap': train_acc - test_acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Test_Acc', ascending=False)

print("\n" + "-" * 80)
print("Performance Summary (Sorted by Test Accuracy)")
print("-" * 80)
print(df_results.to_string(index=False))

print("\n" + "-" * 80)
print("KEY FINDINGS")
print("-" * 80)

best_model = df_results.iloc[0]
print(f"\n Best Overall Model: {best_model['Model']}")
print(f"   Test Accuracy: {best_model['Test_Acc']:.4f} ({best_model['Test_Acc']*100:.2f}%)")
print(f"   Precision (Malignant): {best_model['Precision']:.4f}")
print(f"   Recall (Malignant): {best_model['Recall']:.4f}")
print(f"   F1-Score: {best_model['F1']:.4f}")

least_overfit = df_results.loc[df_results['Gap'].idxmin()]
print(f"\n✓ Least Overfitting: {least_overfit['Model']}")
print(f"   Overfitting Gap: {least_overfit['Gap']:.4f}")

best_recall = df_results.loc[df_results['Recall'].idxmax()]
print(f"\n Best Recall (Cancer Detection): {best_recall['Model']}")
print(f"   Recall: {best_recall['Recall']:.4f} ({best_recall['Recall']*100:.2f}%)")
print(f"   Missed Cancer Cases: {64 - int(best_recall['Recall'] * 64)} out of 64")

print("\n" + "-" * 80)
print("ENSEMBLE METHOD COMPARISON")
print("-" * 80)
print("\nBagging vs Boosting vs Stacking:")
print("  Bagging:")
print("    ✓ Reduces variance, prevents overfitting")
print("    ✓ Parallelizable (fast training)")
print("    ✗ May not reduce bias significantly")
print("\n  Boosting (AdaBoost, GradientBoost):")
print("    ✓ Reduces both bias and variance")
print("    ✓ Often achieves highest accuracy")
print("    ✗ Sequential training (slower)")
print("    ✗ More prone to overfitting than Bagging")
print("\n  Stacking:")
print("    ✓ Combines strengths of diverse models")
print("    ✓ Lowest overfitting (uses CV)")
print("    ✓ Best generalization")
print("    ✗ Most complex implementation")
print("    ✗ Requires careful base model selection")

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("=" * 80)

ML LAB-07: ENSEMBLE LEARNING
Bagging, Boosting, and Stacking Classifiers

LOADING DATASET
--------------------------------------------------------------------------------
Dataset: Breast Cancer Wisconsin (Diagnostic)
Samples: 569
Features: 30
Classes: ['malignant' 'benign']
Class distribution:
  - Malignant (0): 212
  - Benign (1): 357

Training set: 398 samples
Testing set: 171 samples
✓ Features scaled using StandardScaler


EXPERIMENT 1: BAGGING CLASSIFIER
--------------------------------------------------------------------------------

Bagging (Bootstrap Aggregating):
  Concept: Reduce variance by training on different data subsets
  Method:
    1. Create multiple bootstrap samples (random sampling with replacement)
    2. Train a base classifier on each sample
    3. Combine predictions by majority voting
  Benefits:
    - Reduces overfitting
    - Works well with unstable learners (e.g., Decision Trees)
    - Can be parallelized (trains models independently)

Configuration:
  Bas

In [ ]:
"""
ML LAB-07: ENSEMBLE LEARNING
Bagging, Boosting, and Stacking Classifiers

This script implements and compares various ensemble learning techniques:
- Bagging (Bootstrap Aggregating)
- AdaBoost (Adaptive Boosting)
- Gradient Boosting
- Stacking (Stacked Generalization)
- Random Forest (Bonus)

Author: ML Lab
Date: 2026
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data and Preprocessing
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Base Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Ensemble Models
from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    RandomForestClassifier
)

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("=" * 80)
print("ML LAB-07: ENSEMBLE LEARNING")
print("Bagging, Boosting, and Stacking Classifiers")
print("=" * 80)
print()

# ============================================================================
# LOAD AND PREPARE DATA
# ============================================================================
print("LOADING DATASET")
print("-" * 80)

data = load_breast_cancer()
X = data.data
y = data.target

print(f"Dataset: Breast Cancer Wisconsin (Diagnostic)")
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {data.target_names}")
print(f"Class distribution:")
print(f"  - Malignant (0): {sum(y==0)}")
print(f"  - Benign (1): {sum(y==1)}")

# Train-test split (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Feature Scaling (important for KNN in stacking)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print("✓ Features scaled using StandardScaler")

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 1: BAGGING CLASSIFIER
# ============================================================================
print("\nEXPERIMENT 1: BAGGING CLASSIFIER")
print("-" * 80)

print("\nBagging (Bootstrap Aggregating):")
print("  Concept: Reduce variance by training on different data subsets")
print("  Method:")
print("    1. Create multiple bootstrap samples (random sampling with replacement)")
print("    2. Train a base classifier on each sample")
print("    3. Combine predictions by majority voting")
print("  Benefits:")
print("    - Reduces overfitting")
print("    - Works well with unstable learners (e.g., Decision Trees)")
print("    - Can be parallelized (trains models independently)")

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Base Estimator: Decision Tree (unpruned)")
print(f"  Number of Estimators: 100")
print(f"  Bootstrap: True (sampling with replacement)")
print(f"  Max Samples: 100% (use all training data for each bag)")

print("\nTraining Bagging Classifier...")
bagging_model.fit(X_train_scaled, y_train)
y_pred_bagging = bagging_model.predict(X_test_scaled)

bagging_train_acc = accuracy_score(y_train, bagging_model.predict(X_train_scaled))
bagging_test_acc = accuracy_score(y_test, y_pred_bagging)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {bagging_train_acc:.4f} ({bagging_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {bagging_test_acc:.4f} ({bagging_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {bagging_train_acc - bagging_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_bagging = confusion_matrix(y_test, y_pred_bagging)
print(cm_bagging)
print(f"  TN: {cm_bagging[0,0]}, FP: {cm_bagging[0,1]}")
print(f"  FN: {cm_bagging[1,0]}, TP: {cm_bagging[1,1]}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bagging, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 2: BOOSTING CLASSIFIERS
# ============================================================================
print("\nEXPERIMENT 2: BOOSTING CLASSIFIERS")
print("-" * 80)

print("\nBoosting Concept:")
print("  - Trains models sequentially (cannot be parallelized)")
print("  - Each model focuses on correcting previous errors")
print("  - Combines weak learners to create a strong learner")
print("  - Reduces both bias and variance")

# ──────────────────────────────────────────────────────────────────────────
# 2A. AdaBoost (Adaptive Boosting)
# ──────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("2A. AdaBoost (Adaptive Boosting)")
print("─" * 80)

print("\nHow AdaBoost Works:")
print("  1. Initialize: Equal weights for all training samples")
print("  2. Train weak learner on weighted data")
print("  3. Calculate weighted error")
print("  4. Update sample weights:")
print("     - Increase weights of misclassified samples")
print("     - Decrease weights of correctly classified samples")
print("  5. Repeat steps 2-4 for n_estimators")
print("  6. Final prediction: Weighted majority vote")

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # Decision stump
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Base Estimator: Decision Stump (max_depth=1)")
print(f"  Number of Estimators: 100")
print(f"  Learning Rate: 0.5")
print(f"  Algorithm: SAMME.R (real-valued predictions)")

print("\nTraining AdaBoost...")
adaboost_model.fit(X_train_scaled, y_train)
y_pred_adaboost = adaboost_model.predict(X_test_scaled)

adaboost_train_acc = accuracy_score(y_train, adaboost_model.predict(X_train_scaled))
adaboost_test_acc = accuracy_score(y_test, y_pred_adaboost)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {adaboost_train_acc:.4f} ({adaboost_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {adaboost_test_acc:.4f} ({adaboost_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {adaboost_train_acc - adaboost_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_adaboost = confusion_matrix(y_test, y_pred_adaboost)
print(cm_adaboost)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_adaboost, target_names=data.target_names))

# ──────────────────────────────────────────────────────────────────────────
# 2B. Gradient Boosting
# ──────────────────────────────────────────────────────────────────────────
print("\n" + "─" * 80)
print("2B. Gradient Boosting")
print("─" * 80)

print("\nHow Gradient Boosting Works:")
print("  1. Initialize with a simple model (predicts mean/mode)")
print("  2. Calculate residuals (actual - predicted)")
print("  3. Fit new model to predict residuals")
print("  4. Update ensemble: F(x) = F(x) + lr × new_model(x)")
print("  5. Repeat steps 2-4 for n_estimators")
print("  6. Final prediction: Sum of all models with learning rate")

gboost_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Number of Estimators: 100")
print(f"  Learning Rate: 0.1")
print(f"  Max Depth: 3")
print(f"  Subsample: 1.0 (use all samples)")

print("\nTraining Gradient Boosting...")
gboost_model.fit(X_train_scaled, y_train)
y_pred_gboost = gboost_model.predict(X_test_scaled)

gboost_train_acc = accuracy_score(y_train, gboost_model.predict(X_train_scaled))
gboost_test_acc = accuracy_score(y_test, y_pred_gboost)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {gboost_train_acc:.4f} ({gboost_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {gboost_test_acc:.4f} ({gboost_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {gboost_train_acc - gboost_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_gboost = confusion_matrix(y_test, y_pred_gboost)
print(cm_gboost)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gboost, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 3: STACKING CLASSIFIER
# ============================================================================
print("\nEXPERIMENT 3: STACKING CLASSIFIER")
print("-" * 80)

print("\nStacking (Stacked Generalization):")
print("  Concept: Learn to optimally combine diverse models")
print("  Architecture:")
print("    Level 0 (Base Models):")
print("      - Multiple diverse classifiers")
print("      - Each captures different patterns")
print("    Level 1 (Meta-Model):")
print("      - Learns from base model predictions")
print("      - Combines predictions optimally")

print("\nHow Stacking Works:")
print("  1. Split training data for cross-validation")
print("  2. Train base models on CV folds")
print("  3. Generate meta-features (base model predictions)")
print("  4. Train meta-model on meta-features")
print("  5. Final prediction: Meta-model output")

print("\nOur Stacking Architecture:")
print("  Base Models (Level 0):")
print("    1. Decision Tree - Captures non-linear patterns")
print("    2. K-Nearest Neighbors - Instance-based learning")
print("    3. Logistic Regression - Linear decision boundaries")
print("  Meta-Model (Level 1):")
print("    → Logistic Regression - Combines base predictions")

base_learners = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5  # 5-fold cross-validation for generating meta-features
)

print(f"\nConfiguration:")
print(f"  Base Learners: 3 (Decision Tree, KNN, Logistic Regression)")
print(f"  Final Estimator: Logistic Regression")
print(f"  Cross-Validation: 5-fold")
print(f"  Passthrough: False (only use base predictions)")

print("\nTraining Stacking Classifier...")
print("  [Step 1/3] Training base models with 5-fold CV...")
print("  [Step 2/3] Generating meta-features...")
print("  [Step 3/3] Training meta-model...")

stacking_model.fit(X_train_scaled, y_train)
y_pred_stacking = stacking_model.predict(X_test_scaled)

stacking_train_acc = accuracy_score(y_train, stacking_model.predict(X_train_scaled))
stacking_test_acc = accuracy_score(y_test, y_pred_stacking)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {stacking_train_acc:.4f} ({stacking_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {stacking_test_acc:.4f} ({stacking_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {stacking_train_acc - stacking_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_stacking = confusion_matrix(y_test, y_pred_stacking)
print(cm_stacking)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_stacking, target_names=data.target_names))

# Show individual base model performance
print("\n" + "─" * 80)
print("Individual Base Model Performance vs Stacking:")
print("─" * 80)

for name, model in base_learners:
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    model_name = {'dt': 'Decision Tree', 'knn': 'KNN', 'lr': 'Logistic Regression'}[name]
    print(f"  {model_name:20s}: {acc:.4f} ({acc*100:.2f}%)")

print(f"  {'Stacking (Combined)':20s}: {stacking_test_acc:.4f} ({stacking_test_acc*100:.2f}%)")
print("─" * 80)

print("\n" + "=" * 80)

# ============================================================================
# EXPERIMENT 4: RANDOM FOREST (BONUS)
# ============================================================================
print("\nBONUS: RANDOM FOREST CLASSIFIER")
print("-" * 80)

print("\nRandom Forest:")
print("  - Special case of Bagging with Decision Trees")
print("  - Additional randomness: Random feature subset at each split")
print("  - Each tree sees different data AND different features")
print("  - Reduces correlation between trees → better ensemble")

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(f"\nConfiguration:")
print(f"  Number of Estimators: 100")
print(f"  Max Features: sqrt(n_features) ≈ {int(np.sqrt(X.shape[1]))}")
print(f"  Bootstrap: True")

print("\nTraining Random Forest...")
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_test, y_pred_rf)

print("✓ Training completed")
print(f"\nResults:")
print(f"  Training Accuracy: {rf_train_acc:.4f} ({rf_train_acc*100:.2f}%)")
print(f"  Testing Accuracy:  {rf_test_acc:.4f} ({rf_test_acc*100:.2f}%)")
print(f"  Overfitting Gap:   {rf_train_acc - rf_test_acc:.4f}")

print("\nConfusion Matrix:")
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=data.target_names))

print("\n" + "=" * 80)

# ============================================================================
# COMPREHENSIVE COMPARISON
# ============================================================================
print("\nCOMPREHENSIVE MODEL COMPARISON")
print("=" * 80)

models = {
    'Bagging': bagging_model,
    'AdaBoost': adaboost_model,
    'GradientBoost': gboost_model,
    'Stacking': stacking_model,
    'RandomForest': rf_model
}

results = []
for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    y_pred = model.predict(X_test_scaled)

    precision = precision_score(y_test, y_pred, pos_label=0)
    recall = recall_score(y_test, y_pred, pos_label=0)
    f1 = f1_score(y_test, y_pred, pos_label=0)

    results.append({
        'Model': name,
        'Train_Acc': train_acc,
        'Test_Acc': test_acc,
        'Gap': train_acc - test_acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('Test_Acc', ascending=False)

print("\n" + "-" * 80)
print("Performance Summary (Sorted by Test Accuracy)")
print("-" * 80)
print(df_results.to_string(index=False))

print("\n" + "-" * 80)
print("KEY FINDINGS")
print("-" * 80)

best_model = df_results.iloc[0]
print(f"\n Best Overall Model: {best_model['Model']}")
print(f"   Test Accuracy: {best_model['Test_Acc']:.4f} ({best_model['Test_Acc']*100:.2f}%)")
print(f"   Precision (Malignant): {best_model['Precision']:.4f}")
print(f"   Recall (Malignant): {best_model['Recall']:.4f}")
print(f"   F1-Score: {best_model['F1']:.4f}")

least_overfit = df_results.loc[df_results['Gap'].idxmin()]
print(f"\n✓ Least Overfitting: {least_overfit['Model']}")
print(f"   Overfitting Gap: {least_overfit['Gap']:.4f}")

best_recall = df_results.loc[df_results['Recall'].idxmax()]
print(f"\n Best Recall (Cancer Detection): {best_recall['Model']}")
print(f"   Recall: {best_recall['Recall']:.4f} ({best_recall['Recall']*100:.2f}%)")
print(f"   Missed Cancer Cases: {64 - int(best_recall['Recall'] * 64)} out of 64")

print("\n" + "-" * 80)
print("ENSEMBLE METHOD COMPARISON")
print("-" * 80)
print("\nBagging vs Boosting vs Stacking:")
print("  Bagging:")
print("    ✓ Reduces variance, prevents overfitting")
print("    ✓ Parallelizable (fast training)")
print("    ✗ May not reduce bias significantly")
print("\n  Boosting (AdaBoost, GradientBoost):")
print("    ✓ Reduces both bias and variance")
print("    ✓ Often achieves highest accuracy")
print("    ✗ Sequential training (slower)")
print("    ✗ More prone to overfitting than Bagging")
print("\n  Stacking:")
print("    ✓ Combines strengths of diverse models")
print("    ✓ Lowest overfitting (uses CV)")
print("    ✓ Best generalization")
print("    ✗ Most complex implementation")
print("    ✗ Requires careful base model selection")

print("\n" + "=" * 80)
print("EXPERIMENT COMPLETED SUCCESSFULLY!")
print("=" * 80)

ML LAB-07: ENSEMBLE LEARNING
Bagging, Boosting, and Stacking Classifiers

LOADING DATASET
--------------------------------------------------------------------------------
Dataset: Breast Cancer Wisconsin (Diagnostic)
Samples: 569
Features: 30
Classes: ['malignant' 'benign']
Class distribution:
  - Malignant (0): 212
  - Benign (1): 357

Training set: 398 samples
Testing set: 171 samples
✓ Features scaled using StandardScaler


EXPERIMENT 1: BAGGING CLASSIFIER
--------------------------------------------------------------------------------

Bagging (Bootstrap Aggregating):
  Concept: Reduce variance by training on different data subsets
  Method:
    1. Create multiple bootstrap samples (random sampling with replacement)
    2. Train a base classifier on each sample
    3. Combine predictions by majority voting
  Benefits:
    - Reduces overfitting
    - Works well with unstable learners (e.g., Decision Trees)
    - Can be parallelized (trains models independently)

Configuration:
  Bas